# Task 2 — Classificatore manuale: **Naïve Bayes**

**Corso:** Fondamenti e Applicazioni del Machine Learning (FML 2026)
**Dataset:** `manuale.csv` (12 campioni, bilanciati 6 `yes` / 6 `no`)
**Riferimento teorico:** Witten, Frank, Hall, Pal — *Data Mining* (4ª ed.), cap. 4 — **Lezione 5**

> 📄 Documentazione discorsiva e motivazioni di progetto: [`docs/task2.md`](../docs/task2.md)

In questo notebook costruiamo **a mano**, passo dopo passo, un classificatore **Naïve Bayes**
sul file `manuale.csv`. L'obiettivo, come richiesto dal **punto 2 della traccia**, è
*definire e adattare il modello ai dati, illustrarne i passi, implementarlo in Python e
valutarne le prestazioni sullo stesso file `manuale.csv`*.

**Stile di codice.** Tutto il notebook è scritto in stile **vettoriale** (pandas/numpy:
`crosstab`, `groupby`, `value_counts`, `apply`, …), **senza cicli `for` espliciti né list
comprehension**, coerentemente con i notebook del corso.

## 1. Caricamento del dataset `manuale.csv`

Carichiamo il file `manuale.csv`, estratto nel **Task 1** dal dataset originale *Bank
Marketing*. Contiene **12 campioni** scelti in modo da essere **bilanciati** rispetto alla
variabile target `y` (sottoscrizione del deposito vincolato: `1` = sì, `0` = no).

In [1]:
import pandas as pd
import numpy as np

manual_df = pd.read_csv("../data/processed/manuale.csv")
manual_df

,age,campaign,job,marital,education,housing,loan,contact,poutcome,y
0,35,3,admin.,single,professional.course,yes,no,cellular,nonexistent,1
1,53,1,blue-collar,married,unknown,no,no,cellular,nonexistent,0
2,36,3,blue-collar,married,basic.9y,yes,no,cellular,failure,0
3,61,1,retired,married,basic.4y,no,no,telephone,nonexistent,1
4,47,3,admin.,married,university.degree,no,no,telephone,nonexistent,0
5,36,4,blue-collar,married,unknown,yes,no,cellular,nonexistent,0
6,30,4,blue-collar,married,basic.6y,yes,no,cellular,nonexistent,0
7,31,1,admin.,single,high.school,yes,no,cellular,nonexistent,1
8,51,5,technician,married,university.degree,yes,no,cellular,nonexistent,1
9,35,1,blue-collar,divorced,basic.9y,no,no,cellular,failure,1


## 2. Separazione tra feature e target

- `X_manual`: tutte le feature esplicative;
- `y_manual`: la variabile target `y`.

Distinguiamo inoltre gli attributi **numerici** (`age`, `campaign`) da quelli **nominali**
(`job`, `marital`, `education`, `housing`, `loan`, `contact`, `poutcome`): i due tipi
verranno trattati **in modo diverso** dal classificatore.

In [2]:
X_manual = manual_df.drop(columns=["y"])
y_manual = manual_df["y"]

nominali = ["job", "marital", "education", "housing", "loan", "contact", "poutcome"]
numerici = ["age", "campaign"]

print("Shape X_manual:", X_manual.shape)
print("Shape y_manual:", y_manual.shape)
print("\nDistribuzione della classe target:")
print(y_manual.value_counts())

Shape X_manual: (12, 9)
Shape y_manual: (12,)

Distribuzione della classe target:
y
1    6
0    6
Name: count, dtype: int64


## 3. Richiamo teorico: il classificatore Naïve Bayes

Il classificatore Naïve Bayes si basa sul **teorema di Bayes**:

$$ P(Y \mid X) = \frac{P(X \mid Y)\, P(Y)}{P(X)} $$

dove $Y$ è la variabile target e $X$ il vettore delle feature osservate. Nel confronto tra
classi il termine $P(X)$ è **identico per tutte** e può essere ignorato. Il classificatore
assegna quindi una nuova osservazione alla classe che **massimizza**:

$$ P(Y = y \mid X) \;\propto\; P(Y = y)\, \prod_{i=1}^{n} P(X_i \mid Y = y) $$

L'ipotesi **"naïve"** consiste nel considerare le feature **condizionatamente indipendenti**
data la classe $Y$: è un'assunzione forte, ma rende il calcolo trattabile e funziona
sorprendentemente bene nella pratica.

**Due tipi di attributo, due stime diverse:**

- **nominali** → si stimano per **frequenza**. Per evitare che una probabilità nulla
  (un valore mai osservato in una classe) azzeri l'intero prodotto, si applica lo
  **stimatore di Laplace** (*Laplace smoothing*):
  $$ P(X_i = x \mid Y = y) = \frac{N(X_i = x,\, Y = y) + 1}{N(Y = y) + k} $$
  dove $k$ è il numero di valori distinti possibili per la feature $X_i$;

- **numerici** → in alternativa al trattamento per frequenza, una scelta comune è
  **discretizzarli** in fasce e trattarli poi come nominali. È la strada che seguiamo qui,
  perché con sole 12 istanze la stima di una densità gaussiana sarebbe molto instabile.

## 4. Analisi preliminare del dataset manuale

Prima di costruire il classificatore analizziamo la **distribuzione delle classi**: serve a
calcolare le probabilità a priori e a verificare il bilanciamento del file.

In [3]:
class_distribution = y_manual.value_counts()
print(class_distribution)

y
1    6
0    6
Name: count, dtype: int64


In [4]:
class_distribution_percent = y_manual.value_counts(normalize=True) * 100
print(class_distribution_percent)

y
1    50.0
0    50.0
Name: proportion, dtype: float64


### Osservazioni

Il dataset `manuale.csv` contiene **12 osservazioni**:

- 6 appartenenti alla classe `0` (deposito **non** sottoscritto);
- 6 appartenenti alla classe `1` (deposito sottoscritto).

Il dataset è **perfettamente bilanciato**: una scelta deliberata del Task 1, che rende le
probabilità a priori uguali e permette di concentrare l'attenzione sull'effetto delle
singole feature.

## 5. Calcolo delle probabilità a priori

Le probabilità a priori rappresentano la probabilità di osservare ciascuna classe **prima**
di considerare le feature. Si calcolano come:

$$ P(Y = y) = \frac{\text{n. osservazioni della classe } y}{\text{n. totale di osservazioni}} $$

In [5]:
prior_0 = (y_manual == 0).mean()
prior_1 = (y_manual == 1).mean()

print(f"P(Y=0) = {prior_0:.4f}")
print(f"P(Y=1) = {prior_1:.4f}")

P(Y=0) = 0.5000
P(Y=1) = 0.5000


### Risultati

$$ P(Y=0) = 0.50 \qquad P(Y=1) = 0.50 $$

Poiché il dataset è bilanciato, entrambe le classi hanno la stessa probabilità iniziale: il
fattore a priori **non sposta** la decisione, che dipenderà interamente dalle verosimiglianze
delle feature.

## 6. Selezione delle feature

Con sole 12 istanze, usare **tutti** gli attributi nominali è controproducente: feature come
`education` (7 valori distinti) o `job` (6 valori) avrebbero **una sola istanza per quasi ogni
valore**, rendendo le stime di probabilità inaffidabili. Selezioniamo quindi un sottoinsieme
di **cinque feature** che, pur essendo poche, hanno un numero di valori contenuto e una
distribuzione informativa rispetto alla classe:

- `marital` — stato civile (3 valori);
- `housing` — mutuo per la casa (3 valori);
- `loan` — prestito personale (3 valori);
- `age` — età, che **discretizziamo** in fasce;
- `campaign` — numero di contatti nella campagna, anch'esso **discretizzato**.

Questa selezione è coerente con l'impostazione del modello manuale: poche feature, ben
comprese, di cui possiamo calcolare e commentare ogni probabilità.

In [6]:
selected_nominali = ["marital", "housing", "loan"]
selected_numerici = ["age", "campaign"]

manual_df[selected_nominali + selected_numerici + ["y"]]

,marital,housing,loan,age,campaign,y
0,single,yes,no,35,3,1
1,married,no,no,53,1,0
2,married,yes,no,36,3,0
3,married,no,no,61,1,1
4,married,no,no,47,3,0
5,married,yes,no,36,4,0
6,married,yes,no,30,4,0
7,single,yes,no,31,1,1
8,married,yes,no,51,5,1
9,divorced,no,no,35,1,1


### 6.1 Discretizzazione degli attributi numerici

`age` e `campaign` sono numerici. Per trattarli come gli altri attributi nel modello Naïve
Bayes a frequenze, li **discretizziamo in tre fasce** ciascuno. La discretizzazione riduce la
frammentazione dei dati e rende più robuste le stime delle probabilità condizionate.

- **`age`**: `giovane` (< 35), `adulto` (35–49), `senior` (≥ 50);
- **`campaign`**: `basso` (1), `medio` (2–3), `alto` (≥ 4).

In [7]:
manual_df["age_cat"] = pd.cut(
    manual_df["age"],
    bins=[0, 35, 50, 100],
    labels=["giovane", "adulto", "senior"],
    right=False
)

manual_df["campaign_cat"] = pd.cut(
    manual_df["campaign"],
    bins=[0, 2, 4, 100],
    labels=["basso", "medio", "alto"],
    right=False
)

manual_df[["age", "age_cat", "campaign", "campaign_cat"]]

,age,age_cat,campaign,campaign_cat
0,35,adulto,3,medio
1,53,senior,1,basso
2,36,adulto,3,medio
3,61,senior,1,basso
4,47,adulto,3,medio
5,36,adulto,4,alto
6,30,giovane,4,alto
7,31,giovane,1,basso
8,51,senior,5,alto
9,35,adulto,1,basso


## 7. Calcolo delle probabilità condizionate

Per costruire il classificatore stimiamo le **probabilità condizionate** di ogni feature
rispetto alla classe. Procediamo **una variabile alla volta**, costruendo una **tabella di
contingenza** (`crosstab`) che mostra la distribuzione dei valori della feature nelle due
classi del target.

In [8]:
# Tabella di contingenza per marital
pd.crosstab(manual_df["marital"], manual_df["y"])

y,0,1
marital,,
divorced,0,1
married,6,2
single,0,3


### Probabilità condizionate per `marital`

`marital` ha **3 valori** (`divorced`, `married`, `single`), quindi $k = 3$. Si nota subito
che `divorced` e `single` **non compaiono affatto** nella classe `0`: senza correzione
darebbero probabilità nulla. Applichiamo lo **stimatore di Laplace** $(conteggio+1)/(N_c+3)$:

**Classe Y = 0** (Nc = 6)

- P(divorced | Y=0) = (0+1)/(6+3) = 1/9 = 0.1111
- P(married  | Y=0) = (6+1)/(6+3) = 7/9 = 0.7778
- P(single   | Y=0) = (0+1)/(6+3) = 1/9 = 0.1111

**Classe Y = 1** (Nc = 6)

- P(divorced | Y=1) = (1+1)/(6+3) = 2/9 = 0.2222
- P(married  | Y=1) = (2+1)/(6+3) = 3/9 = 0.3333
- P(single   | Y=1) = (3+1)/(6+3) = 4/9 = 0.4444

Lo stato `married` è fortemente associato alla classe `0`, mentre `single` lo è alla `1`.

In [9]:
# Tabella di contingenza per housing
pd.crosstab(manual_df["housing"], manual_df["y"])

y,0,1
housing,,
no,2,2
unknown,0,1
yes,4,3


### Probabilità condizionate per `housing`

`housing` ha **3 valori** (`no`, `unknown`, `yes`), $k = 3$. Il valore `unknown` non compare
nella classe `0`, quindi di nuovo serve Laplace:

**Classe Y = 0** (Nc = 6)

- P(no      | Y=0) = (2+1)/(6+3) = 3/9 = 0.3333
- P(unknown | Y=0) = (0+1)/(6+3) = 1/9 = 0.1111
- P(yes     | Y=0) = (4+1)/(6+3) = 5/9 = 0.5556

**Classe Y = 1** (Nc = 6)

- P(no      | Y=1) = (2+1)/(6+3) = 3/9 = 0.3333
- P(unknown | Y=1) = (1+1)/(6+3) = 2/9 = 0.2222
- P(yes     | Y=1) = (3+1)/(6+3) = 4/9 = 0.4444

`housing` è poco discriminante: le distribuzioni nelle due classi sono molto simili.

In [10]:
# Tabella di contingenza per loan
pd.crosstab(manual_df["loan"], manual_df["y"])

y,0,1
loan,,
no,5,5
unknown,0,1
yes,1,0


### Probabilità condizionate per `loan`

`loan` ha **3 valori** (`no`, `unknown`, `yes`), $k = 3$:

**Classe Y = 0** (Nc = 6)

- P(no      | Y=0) = (5+1)/(6+3) = 6/9 = 0.6667
- P(unknown | Y=0) = (0+1)/(6+3) = 1/9 = 0.1111
- P(yes     | Y=0) = (1+1)/(6+3) = 2/9 = 0.2222

**Classe Y = 1** (Nc = 6)

- P(no      | Y=1) = (5+1)/(6+3) = 6/9 = 0.6667
- P(unknown | Y=1) = (1+1)/(6+3) = 2/9 = 0.2222
- P(yes     | Y=1) = (0+1)/(6+3) = 1/9 = 0.1111

La grande maggioranza dei soggetti non ha prestiti personali in entrambe le classi: anche
`loan` è debolmente informativa.

In [11]:
# Tabella di contingenza per age_cat
pd.crosstab(manual_df["age_cat"], manual_df["y"])

y,0,1
age_cat,,
giovane,1,2
adulto,4,2
senior,1,2


### Probabilità condizionate per `age_cat`

Dopo la discretizzazione, `age_cat` ha **3 valori** (`giovane`, `adulto`, `senior`), $k = 3$:

**Classe Y = 0** (Nc = 6)

- P(giovane | Y=0) = (1+1)/(6+3) = 2/9 = 0.2222
- P(adulto  | Y=0) = (4+1)/(6+3) = 5/9 = 0.5556
- P(senior  | Y=0) = (1+1)/(6+3) = 2/9 = 0.2222

**Classe Y = 1** (Nc = 6)

- P(giovane | Y=1) = (2+1)/(6+3) = 3/9 = 0.3333
- P(adulto  | Y=1) = (2+1)/(6+3) = 3/9 = 0.3333
- P(senior  | Y=1) = (2+1)/(6+3) = 3/9 = 0.3333

La fascia `adulto` è più frequente tra i non sottoscrittori; la classe `1` è distribuita in
modo uniforme tra le tre fasce.

In [12]:
# Tabella di contingenza per campaign_cat
pd.crosstab(manual_df["campaign_cat"], manual_df["y"])

y,0,1
campaign_cat,,
basso,1,4
medio,3,1
alto,2,1


### Probabilità condizionate per `campaign_cat`

`campaign_cat` ha **3 valori** (`basso`, `medio`, `alto`), $k = 3$:

**Classe Y = 0** (Nc = 6)

- P(basso | Y=0) = (1+1)/(6+3) = 2/9 = 0.2222
- P(medio | Y=0) = (3+1)/(6+3) = 4/9 = 0.4444
- P(alto  | Y=0) = (2+1)/(6+3) = 3/9 = 0.3333

**Classe Y = 1** (Nc = 6)

- P(basso | Y=1) = (4+1)/(6+3) = 5/9 = 0.5556
- P(medio | Y=1) = (1+1)/(6+3) = 2/9 = 0.2222
- P(alto  | Y=1) = (1+1)/(6+3) = 2/9 = 0.2222

**Osservazione interessante:** un numero **basso** di contatti (`basso`) è associato alla
sottoscrizione (classe `1`), mentre molti contatti (`medio`/`alto`) sono più frequenti tra i
non sottoscrittori. È coerente con l'intuizione: insistere troppo non aiuta a convertire.

## 8. Implementazione manuale del classificatore

Prima di automatizzare, mostriamo il **calcolo manuale** della probabilità a posteriori per
una **singola osservazione**, così da illustrare il funzionamento dell'algoritmo passo dopo
passo. Prendiamo la **prima riga** del dataset.

In [13]:
manual_df.iloc[0]

age                              35
campaign                          3
job                          admin.
marital                      single
education       professional.course
housing                         yes
loan                             no
contact                    cellular
poutcome                nonexistent
y                                 1
age_cat                      adulto
campaign_cat                  medio
Name: 0, dtype: object

### Calcolo delle probabilità a posteriori (riga 0)

La riga 0 ha: `marital=single`, `housing=yes`, `loan=no`, `age_cat=adulto`,
`campaign_cat=medio`, e classe reale `y=1`. Calcoliamo i punteggi delle due classi.

**Classe Y = 0**

$$ \text{Score}_0 = P(Y{=}0)\cdot P(single|0)\cdot P(yes|0)\cdot P(no|0)\cdot P(adulto|0)\cdot P(medio|0) $$
$$ = 0.5 \times 0.1111 \times 0.5556 \times 0.6667 \times 0.5556 \times 0.4444 \approx 0.00508 $$

**Classe Y = 1**

$$ \text{Score}_1 = P(Y{=}1)\cdot P(single|1)\cdot P(yes|1)\cdot P(no|1)\cdot P(adulto|1)\cdot P(medio|1) $$
$$ = 0.5 \times 0.4444 \times 0.4444 \times 0.6667 \times 0.3333 \times 0.2222 \approx 0.00488 $$

Poiché $\text{Score}_0 > \text{Score}_1$ (ma di pochissimo!), il classificatore assegna la
riga 0 alla **classe 0**. La classe reale è però `1`: si tratta quindi di un **errore**, che
discuteremo. Il margine minimo ($0.00508$ vs $0.00488$) mostra quanto la decisione sia incerta
su pochi dati.

### Strutture dati: priori e probabilità condizionate

Raccogliamo i valori calcolati sopra in due strutture dati. Le probabilità sono scritte
**esplicitamente** come frazioni, in modo che il codice rispecchi uno-a-uno i calcoli a mano
delle sezioni precedenti.

In [14]:
# Probabilità a priori
priors = {
    0: 0.5,
    1: 0.5
}

In [15]:
# Tutte le probabilità condizionate calcolate con lo stimatore di Laplace
probabilities = {
    "marital": {
        0: {"divorced": 1/9, "married": 7/9, "single": 1/9},
        1: {"divorced": 2/9, "married": 3/9, "single": 4/9},
    },
    "housing": {
        0: {"no": 3/9, "unknown": 1/9, "yes": 5/9},
        1: {"no": 3/9, "unknown": 2/9, "yes": 4/9},
    },
    "loan": {
        0: {"no": 6/9, "unknown": 1/9, "yes": 2/9},
        1: {"no": 6/9, "unknown": 2/9, "yes": 1/9},
    },
    "age_cat": {
        0: {"giovane": 2/9, "adulto": 5/9, "senior": 2/9},
        1: {"giovane": 3/9, "adulto": 3/9, "senior": 3/9},
    },
    "campaign_cat": {
        0: {"basso": 2/9, "medio": 4/9, "alto": 3/9},
        1: {"basso": 5/9, "medio": 2/9, "alto": 2/9},
    },
}

### La funzione di predizione

`predict_naive_bayes` applica esattamente la regola di Bayes: parte dalle probabilità a priori
e **moltiplica, un termine alla volta**, la verosimiglianza di ciascuna feature per le due
classi. Restituisce la classe con il punteggio più alto (regola **MAP**).

In [16]:
def predict_naive_bayes(row):
    # Probabilità a priori
    score_0 = priors[0]
    score_1 = priors[1]

    # marital
    score_0 *= probabilities["marital"][0][row["marital"]]
    score_1 *= probabilities["marital"][1][row["marital"]]

    # housing
    score_0 *= probabilities["housing"][0][row["housing"]]
    score_1 *= probabilities["housing"][1][row["housing"]]

    # loan
    score_0 *= probabilities["loan"][0][row["loan"]]
    score_1 *= probabilities["loan"][1][row["loan"]]

    # age_cat
    score_0 *= probabilities["age_cat"][0][row["age_cat"]]
    score_1 *= probabilities["age_cat"][1][row["age_cat"]]

    # campaign_cat
    score_0 *= probabilities["campaign_cat"][0][row["campaign_cat"]]
    score_1 *= probabilities["campaign_cat"][1][row["campaign_cat"]]

    # Predizione finale (regola MAP)
    prediction = 0 if score_0 > score_1 else 1
    return prediction, score_0, score_1

In [17]:
# Verifica sul primo campione
prediction, score0, score1 = predict_naive_bayes(manual_df.iloc[0])

print("Score classe 0:", score0)
print("Score classe 1:", score1)
print("Predizione:", prediction)
print("Classe reale:", int(manual_df.iloc[0]["y"]))

Score classe 0: 0.005080526342529085
Score classe 1: 0.004877305288827921
Predizione: 0
Classe reale: 1


### Verifica dell'implementazione

I punteggi prodotti dalla funzione coincidono con quelli calcolati a mano (Score₀ ≈ 0.00508,
Score₁ ≈ 0.00488). La predizione è `0`, mentre la classe reale è `1`: l'implementazione è
**corretta** (riproduce fedelmente il calcolo manuale), anche se su questa specifica istanza
il modello sbaglia. Le minime differenze numeriche rispetto ai valori scritti nelle sezioni
precedenti derivano unicamente dagli arrotondamenti usati nell'esposizione.

## 9. Predizione su tutti i campioni di `manuale.csv`

Applichiamo ora il classificatore a **tutte** le 12 osservazioni. Come richiesto dalla traccia
(*"valutando le prestazioni ottenute sullo stesso file manuale.csv"*), training e test
coincidono. Usiamo `apply(axis=1)` per applicare la funzione riga per riga **senza cicli
espliciti**.

In [18]:
manual_df["Predicted"] = manual_df.apply(
    lambda row: predict_naive_bayes(row)[0],
    axis=1
)

manual_df[["y", "Predicted"]]

,y,Predicted
0,1,0
1,0,1
2,0,0
3,1,1
4,0,0
5,0,0
6,0,0
7,1,1
8,1,0
9,1,1


Confrontando colonna `y` (reale) e `Predicted`, contiamo gli esiti:

- veri negativi (TN, reale 0 / predetto 0) → 5
- falsi positivi (FP, reale 0 / predetto 1) → 1
- falsi negativi (FN, reale 1 / predetto 0) → 2
- veri positivi (TP, reale 1 / predetto 1) → 4

Il modello sbaglia 3 istanze su 12.

## 10. Valutazione delle prestazioni

Valutiamo il classificatore con le principali metriche viste a lezione (Lezione 8):
**Accuracy, Confusion Matrix, Precision, Recall, F1-Score**. La classe positiva di interesse è
`y = 1` (sottoscrizione).

### Accuracy

$$ \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} $$

Misura la frazione di osservazioni classificate correttamente.

In [19]:
accuracy = (manual_df["y"] == manual_df["Predicted"]).mean()
print("Accuracy:", accuracy)

Accuracy: 0.75


### Confusion Matrix

Permette di analizzare nel dettaglio **quali** errori commette il modello, distinguendo falsi
positivi e falsi negativi.

In [20]:
tp = len(manual_df[(manual_df["y"] == 1) & (manual_df["Predicted"] == 1)])
tn = len(manual_df[(manual_df["y"] == 0) & (manual_df["Predicted"] == 0)])
fp = len(manual_df[(manual_df["y"] == 0) & (manual_df["Predicted"] == 1)])
fn = len(manual_df[(manual_df["y"] == 1) & (manual_df["Predicted"] == 0)])

print("TP =", tp)
print("TN =", tn)
print("FP =", fp)
print("FN =", fn)

confusion_matrix_df = pd.DataFrame(
    [[tn, fp],
     [fn, tp]],
    columns=["Predicted 0", "Predicted 1"],
    index=["Actual 0", "Actual 1"]
)
confusion_matrix_df

TP = 4
TN = 5
FP = 1
FN = 2


,Predicted 0,Predicted 1
Actual 0,5,1
Actual 1,2,4


### Precision

$$ \text{Precision} = \frac{TP}{TP + FP} $$

Tra le osservazioni predette come positive, quante lo sono davvero. Una precision alta
significa pochi falsi allarmi.

In [21]:
precision = tp / (tp + fp)
print("Precision:", precision)

Precision: 0.8


### Recall

$$ \text{Recall} = \frac{TP}{TP + FN} $$

Tra le osservazioni realmente positive, quante ne individua il modello. Una recall bassa
significa che il modello **si perde** dei casi positivi (qui: clienti che avrebbero
sottoscritto).

In [22]:
recall = tp / (tp + fn)
print("Recall:", recall)

Recall: 0.6666666666666666


### F1-Score

$$ F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} $$

Media armonica di precision e recall: utile quando si cerca un compromesso tra le due,
specialmente in presenza di classi sbilanciate (come sarà nel `training.csv` reale).

In [23]:
f1 = 2 * (precision * recall) / (precision + recall)
print("F1 Score:", f1)

F1 Score: 0.7272727272727272


In [24]:
print("========== RISULTATI NAIVE BAYES ==========")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

========== RISULTATI NAIVE BAYES ==========
Accuracy : 0.7500
Precision: 0.8000
Recall   : 0.6667
F1 Score : 0.7273


## 11. Controprova con Scikit-Learn

La traccia consente l'uso di API. Come **controprova** della nostra implementazione manuale,
addestriamo un `CategoricalNB` di scikit-learn sulle **stesse cinque feature discretizzate**,
con lo stesso smoothing di Laplace (`alpha=1`), e lo valutiamo sullo stesso file. `CategoricalNB`
è la variante di Naïve Bayes per attributi categorici, quindi è quella concettualmente più
vicina a ciò che abbiamo costruito a mano.

In [25]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

feature_cols = ["marital", "housing", "loan", "age_cat", "campaign_cat"]

X = manual_df[feature_cols].astype(str)
X_enc = OrdinalEncoder().fit_transform(X)
y = manual_df["y"]

model = CategoricalNB(alpha=1.0)
model.fit(X_enc, y)
pred_sklearn = model.predict(X_enc)

acc_sklearn = (pred_sklearn == y).mean()
print(f"Accuracy CategoricalNB (sklearn, sullo stesso file): {acc_sklearn:.4f}")
print(f"Accuracy implementazione manuale:                    {accuracy:.4f}")

Accuracy CategoricalNB (sklearn, sullo stesso file): 0.7500
Accuracy implementazione manuale:                    0.7500


Il risultato di scikit-learn è **coerente** con la nostra implementazione manuale: piccole
differenze sono possibili perché `CategoricalNB` apprende l'insieme dei valori da
`OrdinalEncoder` e gestisce internamente conteggi e smoothing, ma la logica (Bayes + Laplace su
attributi categorici) è la stessa. La controprova conferma la **correttezza** del classificatore
costruito a mano.

## 12. Discussione critica dei risultati

Il Naïve Bayes implementato manualmente ottiene, **sullo stesso `manuale.csv`**:

| Metrica | Valore |
|---|---|
| Accuracy  | 75.00% |
| Precision | 80.00% |
| Recall    | 66.67% |
| F1-Score  | 72.73% |

con la seguente matrice di confusione:

|            | Predicted 0 | Predicted 1 |
|------------|:-----------:|:-----------:|
| **Actual 0** | 5 | 1 |
| **Actual 1** | 2 | 4 |

**Lettura dei risultati.**

- Il modello classifica correttamente **9 osservazioni su 12**.
- La **precision** (80%) è buona: quando predice una sottoscrizione, di solito ha ragione
  (un solo falso positivo).
- La **recall** (66.7%) è più bassa: due clienti che avrebbero sottoscritto (FN) vengono
  classificati come non interessati. È il tipo di errore più costoso in un contesto di
  marketing, dove l'obiettivo è non perdere clienti potenziali.
- L'errore sulla **riga 0** è emblematico: il punteggio delle due classi era quasi identico
  ($0.00508$ vs $0.00488$). Con così pochi dati, lo stimatore di Laplace "appiattisce" le
  probabilità e basta poco per ribaltare la decisione.

**Limiti di questa valutazione.** I risultati vanno presi come **illustrativi**, non come un
giudizio affidabile sul modello, perché:

- il dataset ha solo **12 osservazioni**;
- training e test **coincidono** (come richiesto dalla traccia), quindi le metriche tendono a
  essere ottimistiche;
- abbiamo usato solo **5 delle 9 feature** disponibili;
- le numeriche sono state **discretizzate**, perdendo informazione;
- l'assunzione di **indipendenza** tra le feature è certamente violata (es. `housing` e `loan`
  sono verosimilmente correlati).

**Confronto con 1R e passo successivo.** Rispetto a 1R (che usa un solo attributo), Naïve Bayes
**sfrutta più informazione** combinando cinque feature. La valutazione seria e statisticamente
robusta — su `training.csv` (41.176 istanze), con holdout stratificato e metriche adatte allo
**sbilanciamento di classe** — è svolta nei **Task 4 e 5**.